# Talking to the Reasoning Engine: The `ask()` Method

This tutorial demonstrates how to interact with the `skforecast-ai` **Reasoning Engine** using the `ask()` method.

The Reasoning Engine acts as a Senior Data Scientist co-pilot. Because it utilizes **Knowledge as Code**, its answers are always grounded in official `skforecast` best practices rather than hallucinated LLM knowledge.

You can interact with the agent in several modes:
1. **General Q&A:** Ask theoretical or API questions without any data context.
2. **Explain Mode:** Pass in raw data and ask the agent to profile it and suggest a strategy.
3. **Results Mode:** Pass a completed `ForecastResult` and ask the agent to interpret the predictions.

**Requirements:** A valid API key for your chosen LLM provider.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from skforecast_ai import ForecastingAssistant

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 4)

# Initialize the assistant with your LLM provider
# assistant = ForecastingAssistant(
#     llm='google:gemini-3-flash-preview',
#     # api_key='your-api-key-here'  # Uncomment and provide your key if not using env variables
# )

assistant = ForecastingAssistant(
    llm='bedrock:eu.anthropic.claude-sonnet-4-6',
    base_url="eu-west-1"
)

## 1. General Q&A (No Data)

You can ask the Reasoning Engine general questions about forecasting theory or the `skforecast` library. The agent will automatically select the relevant skills (Markdown documentation) to ground its answer.

In [4]:
answer_general = assistant.ask(
    prompt="What is the difference between ForecasterRecursive and ForecasterDirect? When should I use each one?"
)
print(answer_general.explanation)

[LLM unavailable] Unable to locate credentials


## 2. Explain Mode (Data Profiling & Planning)

When you pass data directly to `ask()`, the assistant first runs the deterministic Execution Engine (profiling the data and generating a declarative plan). It then passes this state to the Reasoning Engine to explain the strategy it formulated.

Let's load the `h2o_exog` dataset (monthly corticosteroid sales with exogenous variables).

In [ ]:
url = 'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/h2o_exog.csv'
data = pd.read_csv(url)

data['fecha'] = pd.to_datetime(data['fecha'])
data = data.set_index('fecha')
data = data.asfreq('MS')

# Ask the agent to analyze the data and propose a plan
answer_explain = assistant.ask(
    prompt="Explain the forecasting strategy you would use for this dataset. What patterns do you see?",
    data=data,
    target="y",
    date_column="fecha",
    steps=12,
)

print("--- EXPLANATION ---")
print(answer_explain.explanation)

If you inspect the `AskResult` object, you can see that the deterministic engine built the underlying profile and plan that the LLM just explained to you.

In [ ]:
print(f"Task type chosen: {answer_explain.profile.task_type}")
print(f"Forecaster chosen: {answer_explain.plan.forecaster}")
print(f"Estimator chosen: {answer_explain.plan.estimator}")

## 3. Results Mode (Diagnosing Forecasts)

The most powerful way to use the Reasoning Engine is to have it evaluate actual predictions.

First, let's execute a real forecast on our data.

In [ ]:
# Execute the forecast
forecast_result = assistant.forecast(
    data=data,
    target="y",
    date_column="fecha",
    steps=12,
)

# Plot the results
fig, ax = plt.subplots()
data['y'].iloc[-36:].plot(ax=ax, label='Historical Sales')
forecast_result.predictions['pred'].plot(ax=ax, label='Forecast', color='red')
ax.set_title('Sales Forecast')
ax.legend()
plt.show()

Now, we pass the `forecast_result` directly back into the `ask()` method. The agent receives the metrics, the predictions, and the pipeline configuration in its context window.

In [ ]:
answer_results = assistant.ask(
    prompt="Explain these predictions. What trend do you see and what values are expected?",
    forecast_result=forecast_result,
)
print(answer_results.explanation)

## 4. Explicit Skill Overrides (Troubleshooting)

Usually, the agent auto-selects the skills it needs based on your prompt. However, you can explicitly force it to use certain skills. This is highly useful for troubleshooting.

If you hit a Python error in your own code, you can pass the traceback and force the agent to consult the `troubleshooting-common-errors` skill.

In [ ]:
answer_troubleshoot = assistant.ask(
    prompt="I'm getting a ValueError when calling predict(). What should I check?",
    skills=["troubleshooting-common-errors", "complete-api-reference"],
)
print(answer_troubleshoot.explanation)